# rPPG Clinical Model - Training on Google Colab
Training notebook for remote photoplethysmography model on MCD-rPPG dataset with GPU.

In [ ]:
!git clone https://github.com/kpanuragh/facescan.git /content/facescan 2>/dev/null || true
%cd /content/facescan
!git pull origin main

In [ ]:
!pip install -q torch torchvision opencv-python mediapipe scikit-learn pandas scipy datasets huggingface-hub fastapi uvicorn pydantic pytest
print('✓ Dependencies installed')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
!python -m pytest tests/test_preprocessing.py -v --tb=short

In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    print('⚠️ HF_TOKEN not found in Colab Secrets!')
    print('Add it: click 🔑 icon → Add secret HF_TOKEN → paste your token')
else:
    from huggingface_hub import login
    login(token=hf_token)
    print('✓ Logged into HuggingFace')

In [ ]:
import sys
sys.path.insert(0, '/content/facescan')

from src.preprocessing.video_loader import VideoLoader
loader = VideoLoader()

print('Loading MCD-rPPG dataset from HuggingFace...')
try:
    dataset = loader.load_dataset()
    print(f'✓ Real dataset loaded: {len(dataset)} samples')
except Exception as e:
    print(f'⚠️ Real dataset error: {type(e).__name__}')
    print(f'Details: {str(e)[:200]}...')
    print('\nUsing synthetic dataset for testing...')
    
    import torch
    from torch.utils.data import TensorDataset
    X = torch.randn(100, 30, 3, 128, 128)
    y_hr = torch.randint(50, 120, (100,)).float()
    y_rr = torch.randint(10, 25, (100,)).float()
    y_spo2 = torch.randint(93, 101, (100,)).float()
    dataset = TensorDataset(X, y_hr, y_rr, y_spo2)
    print(f'✓ Synthetic dataset created: {len(dataset)} samples')

In [ ]:
from src.preprocessing.dataset_builder import create_splits
train_idx, val_idx, test_idx = create_splits(dataset)
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}')

In [ ]:
from src.models.architecture import rPPGModel
from src.config import FEATURE_DIM, HIDDEN_DIM, DEVICE
model = rPPGModel(feature_dim=FEATURE_DIM, hidden_dim=HIDDEN_DIM).to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'✓ Model: {params:,} parameters on {DEVICE}')

In [ ]:
from src.training.trainer import Trainer
from src.config import BATCH_SIZE, LEARNING_RATE, EPOCHS, EARLY_STOPPING_PATIENCE, CHECKPOINTS_DIR
import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)
trainer = Trainer(model, DEVICE, early_stopping_patience=EARLY_STOPPING_PATIENCE, checkpoint_dir=CHECKPOINTS_DIR)

print(f'✓ Trainer initialized')
print(f'  Epochs: {EPOCHS}')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Learning rate: {LEARNING_RATE}')

In [ ]:
print('Starting training...')
history = trainer.train(
    train_indices=train_idx,
    val_indices=val_idx,
    dataset=dataset,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)
print('✓ Training complete!')

In [ ]:
import json
from src.config import RESULTS_DIR

results = {'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LEARNING_RATE}
with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(results, f)

torch.save(model.state_dict(), RESULTS_DIR / 'model.pt')
print('✓ Results and model saved!')